# Procesado NLP variables textuales

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Modelos para generar los embeddings
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

In [3]:
# Reducción de componentes
from sklearn.decomposition import PCA
# Estandarizacion
from sklearn.preprocessing import StandardScaler

In [4]:
# Modelos estadisticas
from sklearn.cluster import KMeans
from scipy.stats import spearmanr

In [5]:
# Graficos
import plotly.express as px

In [6]:
# Modelo de embeddings multilingue
MODEL = "intfloat/multilingual-e5-small"

In [7]:
df_tablon = pd.read_parquet('../data/processed/final/tablon_nlp_base_v3.parquet')

In [8]:
df_tablon.head(1)

,pub,ops_pub_date,roi_proxy,roi_class,ops_n_applicants,ops_n_inventors,ops_n_ipc,ops_n_cpc,ops_ipc_main,family_size_simple,...,years_since_first_legal,years_since_last_legal,alive_span_days,alive_years,events_per_year,is_active_recent,roi_proxy_v2,roi_proxy_v3,cost_proxy_v2,roi_class_v3
0,EP4297217A2,20231227,1.395468,2,2,10,0,20,None,16,...,2.108145,0.52293,579,1.585216,10.681126,1,3.095396,4.446525,4.418429,2


In [9]:
df_tablon.columns

Index(['pub', 'ops_pub_date', 'roi_proxy', 'roi_class', 'ops_n_applicants',
       'ops_n_inventors', 'ops_n_ipc', 'ops_n_cpc', 'ops_ipc_main',
       'family_size_simple', 'family_country_count_simple',
       'family_kinds_count_simple', 'legal_events_count', 'legal_codes_unique',
       'legal_first_date', 'legal_last_date', 'legal_has_grant_like',
       'legal_has_opposition_like', 'ops_pub_date_biblio',
       'ops_n_applicants_biblio', 'ops_n_inventors_biblio', 'ops_n_cpc_biblio',
       'ops_n_ipc_biblio', 'ops_ipc_main_biblio', 'ops_title', 'ops_abstract',
       'biblio_priority_count', 'biblio_priority_date_min',
       'biblio_priority_year', 'biblio_application_date',
       'biblio_application_year', 'biblio_publication_date',
       'biblio_publication_year', 'biblio_kind', 'biblio_title',
       'biblio_abstract', 'party_applicant_count', 'party_inventor_count',
       'party_applicant_country_list', 'party_applicant_country_count',
       'party_applicant_country_main'

## NLP Feature Engineering — Semantic Signals for Patent ROI

Este cuaderno tiene como objetivo generar **características semánticas derivadas del texto de las patentes** mediante técnicas de *Natural Language Processing* (NLP), con el fin de evaluar su **contribución incremental** al modelo estructural de ROI desarrollado previamente.

A diferencia del análisis exploratorio estructural, el texto no se utiliza directamente como predictor, sino como fuente para la construcción de **features semánticas agregadas, continuas e interpretables**, tales como medidas de similitud, novedad o pertenencia a clústeres tecnológicos.

El enfoque adoptado sigue los siguientes principios metodológicos:
- El NLP se emplea exclusivamente como **ingeniería de variables**, no como sustituto del modelo estructural.
- Las señales semánticas se evalúan por su **capacidad de mejorar métricas predictivas** (MAE, RMSE) respecto al baseline.
- Se priorizan representaciones compactas (centroides, distancias, componentes principales) frente a embeddings de alta dimensionalidad sin control.

En esta primera fase, el análisis se centra en el texto del *abstract*, por su mayor estabilidad, menor longitud y mejor relación señal-ruido. El uso de *claims* y *description* se evaluará en fases posteriores.

### Fuente textual utilizada

En esta fase se emplea exclusivamente el campo `text_abstract`, al considerarse la fuente textual más adecuada para capturar el núcleo técnico de la invención con un coste computacional reducido. Los campos `text_claims` y `text_description` se reservan para análisis posteriores debido a su mayor longitud y complejidad.

In [10]:
TEXT_COL = "text_abstract"
df = df_tablon.copy()

# Breadth proxy
df["tech_breadth_proxy"] = df.get("ops_n_cpc", np.nan)
df["tech_breadth_proxy"] = df["tech_breadth_proxy"].fillna(df.get("ops_n_ipc", np.nan))
df["tech_breadth_proxy"] = np.log1p(df["tech_breadth_proxy"])

# 🔑 Asegurar un año de publicación en df ANTES del subset
# Prioridad: biblio_publication_year -> year(ops_pub_date) -> year(biblio_publication_date) -> year(ops_pub_date_biblio)
y = df.get("biblio_publication_year", np.nan)

if "ops_pub_date" in df.columns:
    y = pd.to_datetime(df["ops_pub_date"], errors="coerce").dt.year.fillna(y)

if "biblio_publication_date" in df.columns:
    y = pd.to_datetime(df["biblio_publication_date"], errors="coerce").dt.year.fillna(y)

if "ops_pub_date_biblio" in df.columns:
    y = pd.to_datetime(df["ops_pub_date_biblio"], errors="coerce").dt.year.fillna(y)

df["pub_year"] = y.astype("Int64")

# Ahora sí, subset (incluye pub_year)
df_nlp = df[
    ["pub", "pub_year", "roi_proxy_v3", "roi_class_v3",
     "tech_breadth_proxy", "log_family_size", TEXT_COL]
].dropna(subset=[TEXT_COL]).copy()

In [11]:
df_nlp = df[
    [
        "pub",
        "roi_proxy_v3",
        "roi_class_v3",
        "tech_breadth_proxy",
        "log_family_size",
        TEXT_COL,
    ]
].dropna(subset=[TEXT_COL]).copy()

texts = df_nlp[TEXT_COL].astype(str).tolist()

In [12]:
# Pruebaas Generacion de embeddings
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)

def average_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

@torch.inference_mode()
def embed(texts):
    batch = tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors="pt")
    out = model(**batch)
    emb = average_pool(out.last_hidden_state, batch["attention_mask"])
    emb = F.normalize(emb, p=2, dim=1)
    return emb.cpu().numpy()

queries  = ["query: método para priorizar patentes por ROI", "query: fibrosis sistémica progresión"]
passages = ["passage: este documento describe un proxy de ROI basado en eventos legales...",
            "passage: estudio sobre esclerosis sistémica y fibroblastos..."]

q_emb = embed(queries)
p_emb = embed(passages)
scores = q_emb @ p_emb.T
print(scores)

[[0.84550536 0.8048539 ]
 [0.826836   0.8825865 ]]


In [13]:
# Calibracion modelo distintos idiomas
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
emb = model.encode(
    ["Un texto en español", "A text in English", "Un texte en français"],
    normalize_embeddings=True
)

/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
# Generamos embbeddings del feature "text_abstract" (descripción de la patente, resumen)
texts = df_nlp["text_abstract"].astype(str).tolist()

X_emb = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # 🔑 CLAVE
)

X_emb.shape


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

(200, 384)

In [15]:
# Calculamos centroide global para realizar el clustering (similitud entre abstrats vs ROI)
centroid_all = X_emb.mean(axis=0)
centroid_all = centroid_all / np.linalg.norm(centroid_all)

In [16]:
# Distancia coseno = 1 - similitud coseno
semantic_novelty = 1 - np.dot(X_emb, centroid_all)
df_nlp["semantic_novelty"] = semantic_novelty

In [17]:
# Similitud por alto ROI
HIGH_ROI_CLASS = 2
mask_high = df_nlp["roi_class_v3"] == HIGH_ROI_CLASS
mask_high.mean()
centroid_high = X_emb[mask_high].mean(axis=0)
centroid_high = centroid_high / np.linalg.norm(centroid_high)
sim_to_high_roi = np.dot(X_emb, centroid_high)
df_nlp["sim_to_high_roi"] = sim_to_high_roi

# DIstancia a ROI Alto
dist_to_high_roi = 1 - sim_to_high_roi
df_nlp["dist_to_high_roi"] = dist_to_high_roi


In [18]:
# Reducimos componentes embeddings
pca = PCA(n_components=64, random_state=42)
X_emb_pca = pca.fit_transform(X_emb)

X_emb_pca.shape

(200, 64)

In [19]:
# Aplicamos el clustering
kmeans = KMeans(
    n_clusters=20,
    random_state=42,
    n_init=10
)

semantic_cluster = kmeans.fit_predict(X_emb_pca)
df_nlp["semantic_cluster"] = semantic_cluster

In [20]:
# Distancia al centroide del cluster (patentes ROI_class 2)
cluster_centroids = kmeans.cluster_centers_

dist_to_cluster = np.linalg.norm(
    X_emb_pca - cluster_centroids[semantic_cluster],
    axis=1
)

df_nlp["dist_to_cluster"] = dist_to_cluster

In [21]:
# Aplicación al dataset
nlp_feature_cols = [
    "pub",
    "semantic_novelty",
    "sim_to_high_roi",
    "dist_to_high_roi",
    "semantic_cluster",
    "dist_to_cluster"
]

df_nlp_features = df_nlp[nlp_feature_cols].copy()


In [22]:
df_nlp_features.head()

,pub,semantic_novelty,sim_to_high_roi,dist_to_high_roi,semantic_cluster,dist_to_cluster
0,EP4297217A2,0.340721,0.662029,0.337971,19,0.622704
1,EP4295725A2,0.416563,0.583149,0.416851,4,0.639498
2,EP4296899A2,0.432033,0.561396,0.438604,18,0.757298
3,EP4297537A2,0.533930,0.460546,0.539454,4,0.726105
4,EP4297104A2,0.499019,0.484272,0.515728,10,0.619419


### Features semánticas generadas

A partir de embeddings multilingües del abstract se han construido señales semánticas compactas y interpretables. En particular, la similitud con el centroide de patentes de alto ROI actúa como proxy de alineación tecnológica con áreas históricamente rentables, mientras que la distancia al centroide global captura un grado de novedad semántica. El clustering no supervisado permite identificar áreas tecnológicas latentes y medir la representatividad de cada patente dentro de su dominio semántico.


## Features semánticas derivadas de embeddings

A partir de los *embeddings* multilingües generados sobre el texto del *abstract*, se han construido un conjunto reducido de variables semánticas continuas e interpretables. Estas variables no representan directamente el contenido textual, sino **relaciones geométricas en el espacio semántico**, que permiten capturar alineación tecnológica, novedad y estructura latente del dominio.

### 1. `semantic_novelty`

Esta variable mide la **distancia semántica** entre el embedding de una patente y el centroide global del conjunto de patentes analizadas. Dado que los embeddings están normalizados, esta distancia se interpreta como una distancia coseno.

- Valores bajos indican patentes semánticamente “típicas”, cercanas al contenido medio del conjunto.
- Valores altos indican patentes semánticamente raras o poco frecuentes, potencialmente asociadas a enfoques innovadores o altamente especializados.

Desde un punto de vista conceptual, `semantic_novelty` actúa como un **proxy de novedad semántica**, sin implicar necesariamente mayor valor económico.

---

### 2. `sim_to_high_roi`

Esta variable representa la **similitud coseno** entre el embedding de una patente y el centroide semántico de las patentes clasificadas como de alto ROI (`roi_class_v3 = 2`).

- Valores altos indican que el contenido técnico de la patente es semánticamente cercano a áreas tecnológicas históricamente asociadas a alto retorno económico.
- Valores bajos indican una menor alineación con dichos dominios.

Esta feature captura un concepto de **alineación tecnológica con áreas rentables**, y constituye una de las señales semánticas más directamente interpretables desde el punto de vista económico.

---

### 3. `dist_to_high_roi`

Definida como la distancia coseno complementaria a `sim_to_high_roi`, esta variable mide cuán alejada está una patente del núcleo semántico de las patentes de alto ROI.

- Valores bajos indican cercanía semántica al núcleo de alto ROI.
- Valores altos indican mayor distancia y, por tanto, menor alineación.

Aunque conceptualmente equivalente a `sim_to_high_roi`, su inclusión permite explorar posibles relaciones no lineales o efectos asimétricos en el modelado.

---

### 4. `semantic_cluster`

Esta variable identifica el **clúster semántico** asignado a cada patente mediante un algoritmo de *clustering* no supervisado (K-Means) aplicado sobre una representación reducida (PCA) de los embeddings.

Cada clúster puede interpretarse como un **dominio tecnológico latente**, definido únicamente por similitud semántica del texto y sin información económica explícita.

Esta variable es útil para:
- segmentar el espacio tecnológico,
- analizar patrones de ROI por dominio,
- o introducir efectos de grupo de forma controlada.

---

### 5. `dist_to_cluster`

Esta variable mide la distancia de una patente al centroide del clúster semántico al que pertenece.

- Valores bajos indican patentes representativas del dominio tecnológico del clúster.
- Valores altos indican patentes periféricas, híbridas o transversales.

Desde una perspectiva interpretativa, esta feature captura el **grado de centralidad semántica** dentro de un dominio tecnológico.

---

### Interpretación conjunta y uso en el modelo

Estas features semánticas se incorporan al modelo únicamente si aportan una mejora incremental respecto al baseline estructural. Su función principal es complementar las variables legales, familiares y tecnológicas, proporcionando información sobre **alineación, novedad y estructura semántica** que no está capturada por métricas bibliográficas tradicionales.

En ningún caso estas variables se interpretan de forma causal directa, sino como **señales auxiliares** que enriquecen el espacio de representación del modelo de ROI.


In [23]:
# Correlación variabes obtendas y ROI
nlp_vars = [
    "semantic_novelty",
    "sim_to_high_roi",
    "dist_to_high_roi",
    "dist_to_cluster"
]

corr_results = []

for v in nlp_vars:
    rho, p = spearmanr(df_nlp[v], df_nlp["roi_proxy_v3"], nan_policy="omit")
    corr_results.append({
        "variable": v,
        "rho_spearman": rho,
        "p_value": p
    })

df_corr = pd.DataFrame(corr_results)
df_corr

,variable,rho_spearman,p_value
0,semantic_novelty,0.122562,0.083818
1,sim_to_high_roi,0.017858,0.801824
2,dist_to_high_roi,-0.017858,0.801824
3,dist_to_cluster,0.055486,0.435167


In [24]:
fig = px.bar(
    df_corr,
    x="variable",
    y="rho_spearman",
    color="rho_spearman",
    color_continuous_scale="RdBu",
    range_color=[-1, 1],
    title="Spearman Correlation: NLP Features vs ROI",
    labels={"rho_spearman": "Spearman ρ"}
)

fig.update_layout(template="plotly_white")
fig.show()

In [25]:
fig = px.scatter(
    df_nlp,
    x="semantic_novelty",
    y="roi_proxy_v3",
    opacity=0.5,
    trendline="lowess",
    title="ROI vs Semantic Novelty"
)
fig.update_layout(template="plotly_white")
fig.show()

In [26]:
fig = px.box(
    df_nlp,
    x="roi_class_v3",
    y="sim_to_high_roi",
    points="outliers",
    title="Semantic Similarity to High-ROI by ROI Class"
)
fig.update_layout(template="plotly_white")
fig.show()


## Análisis exploratorio de las variables semánticas

El análisis exploratorio de las variables derivadas de embeddings permite evaluar su relación con el retorno económico sin asumir relaciones lineales ni efectos causales directos.

### Correlación global con el ROI

La correlación de Spearman entre las variables semánticas y el ROI muestra asociaciones débiles en términos absolutos, con valores de ρ generalmente inferiores a 0.15. Este resultado es esperable en un contexto de patentes, donde el valor económico depende de múltiples factores legales, estratégicos y temporales.

No obstante, se observa que:
- `semantic_novelty` presenta una correlación positiva ligera con el ROI, sugiriendo que patentes con contenido técnico menos convencional tienden, en promedio, a mostrar retornos ligeramente superiores.
- `dist_to_cluster` también muestra una correlación positiva moderada, lo que indica que las patentes semánticamente periféricas dentro de su dominio tecnológico pueden estar asociadas a mayores niveles de retorno.
- Las variables directamente derivadas de la similitud a alto ROI (`sim_to_high_roi` y `dist_to_high_roi`) muestran correlaciones globales próximas a cero, lo que sugiere que su efecto no es estrictamente monotónico a nivel global.

---

### Relación entre novedad semántica y ROI

El gráfico de dispersión entre `semantic_novelty` y el ROI muestra una nube de puntos con elevada dispersión y una tendencia suavizada prácticamente plana. Este resultado indica que la novedad semántica, considerada de forma aislada, no constituye un predictor fuerte del retorno económico.

Este comportamiento es consistente con la hipótesis de que la innovación radical conlleva tanto oportunidades como riesgos, y que la novedad por sí sola no garantiza valor económico.

---

### Similitud semántica a alto ROI por clase de ROI

La comparación de la variable `sim_to_high_roi` por clases discretas de ROI muestra una ligera elevación del nivel central para la clase de alto ROI (`roi_class_v3 = 2`). Aunque las distribuciones presentan un solapamiento considerable, este desplazamiento sugiere que las patentes de alto retorno tienden, en promedio, a estar semánticamente más próximas entre sí.

Este resultado refuerza la idea de que existen áreas tecnológicas coherentes asociadas a mayores retornos, aunque dicha coherencia no es suficiente por sí sola para discriminar patentes individuales.


# Otras features semánticas

In [27]:
# Interacciones semánticas simples (alto valor, bajo coste)
df_nlp["novelty_x_breadth"] = (
    df_nlp["semantic_novelty"] * df_nlp["tech_breadth_proxy"]
)

df_nlp["novelty_x_family"] = (
    df_nlp["semantic_novelty"] * df_nlp["log_family_size"]
)

In [28]:
# NLP + tiempo
df_nlp["novelty_x_age"] = (
    df_nlp["semantic_novelty"] * (2025 - df["pub_year"])
)

In [29]:
# Correlacion con ROI
interaction_vars = [
    "novelty_x_breadth",
    "novelty_x_family",
    "novelty_x_age"
]

rows = []
for v in interaction_vars:
    rho, p = spearmanr(df_nlp[v], df_nlp["roi_proxy_v3"], nan_policy="omit")
    rows.append({
        "variable": v,
        "rho_spearman": rho,
        "p_value": p
    })

df_corr_interactions = pd.DataFrame(rows)
df_corr_interactions

,variable,rho_spearman,p_value
0,novelty_x_breadth,0.378530,3.262900e-08
1,novelty_x_family,0.467807,2.861686e-12
2,novelty_x_age,0.122562,8.381845e-02


In [30]:
fig = px.bar(
    df_corr_interactions,
    x="variable",
    y="rho_spearman",
    color="rho_spearman",
    color_continuous_scale="RdBu",
    range_color=[-1, 1],
    title="Spearman Correlation: NLP Interaction Features vs ROI",
    labels={"rho_spearman": "Spearman ρ"}
)

fig.update_layout(template="plotly_white")
fig.show()

In [31]:
fig = px.scatter(
    df_nlp,
    x="novelty_x_breadth",
    y="roi_proxy_v3",
    opacity=0.5,
    trendline="lowess",
    title="ROI vs Semantic Novelty × Technological Breadth"
)
fig.update_layout(template="plotly_white")
fig.show()


In [32]:
fig = px.scatter(
    df_nlp,
    x="novelty_x_family",
    y="roi_proxy_v3",
    opacity=0.5,
    trendline="lowess",
    title="ROI vs Semantic Novelty × Family Size"
)
fig.update_layout(template="plotly_white")
fig.show()


In [33]:
fig = px.scatter(
    df_nlp,
    x="novelty_x_family",
    y="roi_proxy_v3",
    opacity=0.5,
    trendline="lowess",
    title="ROI vs Semantic Novelty × Family Size"
)
fig.update_layout(template="plotly_white")
fig.show()


In [34]:
fig = px.box(
    df_nlp,
    x="roi_class_v3",
    y="novelty_x_breadth",
    points="outliers",
    title="Semantic Novelty × Breadth by ROI Class"
)
fig.update_layout(template="plotly_white")
fig.show()


## Análisis exploratorio de interacciones semántico-estructurales

Con el objetivo de capturar efectos de moderación entre la novedad semántica y dimensiones estructurales clave de la patente, se han analizado tres variables de interacción: `novelty_x_breadth`, `novelty_x_family` y `novelty_x_age`.

Estas variables combinan la información semántica derivada del texto con características tecnológicas, estratégicas y temporales, permitiendo modelar relaciones no lineales que no pueden capturarse mediante variables aisladas.

---

### Correlación con el ROI

El análisis de correlación de Spearman muestra que las interacciones presentan asociaciones sustancialmente más fuertes con el ROI que las variables semánticas simples:

- `novelty_x_family` muestra la correlación más elevada, lo que indica que la novedad semántica es más valiosa cuando la patente se explota a través de una familia amplia.
- `novelty_x_breadth` presenta una correlación positiva clara, sugiriendo que la novedad genera mayor retorno cuando se sitúa en tecnologías con mayor diversidad clasificatoria.
- `novelty_x_age` muestra una correlación positiva más débil, indicando que el efecto temporal de la novedad existe, pero es secundario frente a los factores tecnológicos y estratégicos.

Estos resultados confirman que la contribución de la novedad semántica al retorno económico depende de su contexto de explotación.

---

### Relación funcional con el ROI

Los gráficos de dispersión muestran tendencias crecientes claras entre el ROI y las interacciones `novelty_x_breadth` y `novelty_x_family`, con pendientes positivas y una dispersión moderada. La ausencia de saturaciones abruptas o discontinuidades refuerza la idoneidad de estas variables para modelos lineales regularizados.

En particular, la interacción con el tamaño de la familia muestra una pendiente más pronunciada, lo que apunta a un efecto amplificador de la estrategia internacional sobre el valor de la novedad técnica.

---

### Distribución por clases de ROI

El análisis por clases discretas de ROI revela un desplazamiento sistemático de las distribuciones de `novelty_x_breadth` hacia valores más altos en la clase de alto ROI. Aunque existe solapamiento entre clases, el patrón observado es consistente con la hipótesis de que las patentes de mayor retorno combinan novedad técnica con amplitud tecnológica y capacidad de explotación.

---

### Interpretación global

Las variables de interacción analizadas capturan efectos de segundo orden que no pueden representarse mediante variables semánticas o estructurales de forma independiente. Su comportamiento estable y su asociación consistente con el ROI las convierten en candidatas adecuadas para su inclusión en el modelo, siempre bajo un esquema de regularización que controle posibles redundancias.


In [35]:
# Standarización de todas la interaciones
scaler = StandardScaler()
df_nlp[["novelty_x_family", "novelty_x_breadth"]] = scaler.fit_transform(
    df_nlp[["novelty_x_family", "novelty_x_breadth"]]
)

### Features para el modelo
Nos llevamos las siguientes variables al modelo `novelty_x_family`, `novelty_x_breadth`, `semantic_novelty` y `sim_to_high_roi`

## Enriquecimiento NLP adicional a partir de texto disponible

Además de las variables semánticas derivadas de embeddings, se han identificado un conjunto reducido de señales adicionales basadas exclusivamente en la disponibilidad, estructura y coherencia del texto de la patente.

En primer lugar, se incorporan variables binarias que capturan la completitud documental (`text_available`, `text_has_claims`, `text_has_description`), las cuales actúan como proxies de madurez administrativa y calidad de la información técnica disponible.

En segundo lugar, se consideran ratios entre longitudes de distintos bloques textuales, que permiten caracterizar el estilo de redacción y la complejidad jurídica de la patente sin recurrir a análisis lingüísticos pesados. Estas variables capturan relaciones relativas entre título, abstract y claims, más estables que las longitudes absolutas.

Finalmente, cuando el texto de las claims está disponible, se introduce una medida de coherencia semántica entre el abstract y las reivindicaciones, calculada como similitud entre sus embeddings. Esta variable permite aproximar la alineación entre el núcleo técnico descrito y la estrategia de protección jurídica adoptada.

Estas variables se integran únicamente si aportan mejora incremental al modelo regularizado, y se interpretan como señales auxiliares que complementan las dimensiones estructurales y semánticas previamente analizadas.


# Nuevas features basadas en NLP y texto

In [36]:
# =========================
# 1) Helpers de texto
# =========================

def _clean_text(s):
    """Normaliza texto: NaN si vacío; si no string strip."""
    if pd.isna(s):
        return np.nan
    s = str(s)
    s = " ".join(s.split())  # colapsa espacios
    s = s.strip()
    return s if s else np.nan

def _text_len_chars(s):
    """Longitud en caracteres (NaN si no hay texto)."""
    if pd.isna(s):
        return np.nan
    return len(s)

def _safe_div(num, den):
    """División segura: NaN si den es 0 o NaN."""
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return num / den

In [37]:
# =========================
# 2) Limpieza mínima y flags
# =========================

text_cols = ["text_title", "text_abstract", "text_claims", "text_description"]
for c in text_cols:
    df[c] = df[c].apply(_clean_text)

# (opcional) recalcular flags por robustez
df["text_has_claims"] = df["text_claims"].notna().astype(int)
df["text_has_description"] = df["text_description"].notna().astype(int)
df["text_available"] = (
    df["text_abstract"].notna() |
    df["text_has_claims"].astype(bool) |
    df["text_has_description"].astype(bool)
).astype(int)

In [38]:
# =========================
# 3) Longitudes (caracteres) + log1p
# =========================

df["title_len"] = df["text_title"].apply(_text_len_chars)
df["abstract_len"] = df["text_abstract"].apply(_text_len_chars)
df["claims_len"] = df["text_claims"].apply(_text_len_chars)
df["description_len"] = df["text_description"].apply(_text_len_chars)

# Logs (robustos para colas largas)
for c in ["title_len", "abstract_len", "claims_len", "description_len"]:
    df[f"log_{c}"] = np.log1p(df[c])



In [39]:

# =========================
# 4) Ratios (más informativos que longitudes absolutas)
# =========================

df["abstract_to_title_ratio"] = [
    _safe_div(a, t) for a, t in zip(df["abstract_len"], df["title_len"])
]

df["claims_to_abstract_ratio"] = [
    _safe_div(cl, ab) for cl, ab in zip(df["claims_len"], df["abstract_len"])
]

df["description_to_claims_ratio"] = [
    _safe_div(ds, cl) for ds, cl in zip(df["description_len"], df["claims_len"])
]

# Ratios en log (opcional; útil si hay colas muy largas)
for c in ["abstract_to_title_ratio", "claims_to_abstract_ratio", "description_to_claims_ratio"]:
    df[f"log_{c}"] = np.log1p(df[c])



In [40]:
# =========================
# 5) Embeddings y similitud claims ↔ abstract
# =========================
# Requiere:
#   pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def embed_texts(texts, batch_size=32):
    """Embeddings normalizados (cosine = dot)."""
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

# Para claims↔abstract necesitamos filas donde existan ambos textos
mask_ca = df["text_claims"].notna() & df["text_abstract"].notna()
df_ca = df.loc[mask_ca, ["pub", "text_claims", "text_abstract"]].copy()

# (opcional) recorte de claims muy largas para reducir coste
# Ajusta a tu gusto: si claims son enormes, recortar ayuda (sin "limpieza" agresiva)
MAX_CLAIMS_CHARS = 8000
df_ca["text_claims_cut"] = df_ca["text_claims"].str.slice(0, MAX_CLAIMS_CHARS)

# Embeddings
emb_abs = embed_texts(df_ca["text_abstract"].tolist())
emb_cla = embed_texts(df_ca["text_claims_cut"].tolist())

# Similitud coseno (dot product, porque están normalizados)
sim_claims_abstract = np.sum(emb_abs * emb_cla, axis=1)

# Volcar al df principal
df.loc[mask_ca, "sim_claims_abstract"] = sim_claims_abstract

# Distancia complementaria (opcional)
df.loc[mask_ca, "dist_claims_abstract"] = 1 - df.loc[mask_ca, "sim_claims_abstract"]



/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.



Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [41]:
# =========================
# 6) Features derivadas extra (opcionales, útiles)
# =========================

# Indicadores de "desalineación" (claims amplias vs abstract)
# (si sim es baja y claims_to_abstract_ratio alta, puede indicar claims estratégicas)
df["claims_strategy_proxy"] = df["dist_claims_abstract"] * df["log_claims_to_abstract_ratio"]

# Control: cuando no hay claims, lo dejamos NaN o 0 según prefieras
# df["claims_strategy_proxy"] = df["claims_strategy_proxy"].fillna(0)


# =========================
# 7) Lista final de features textuales nuevas (para inspección rápida)
# =========================

new_text_features = [
    "title_len", "abstract_len", "claims_len", "description_len",
    "log_title_len", "log_abstract_len", "log_claims_len", "log_description_len",
    "abstract_to_title_ratio", "claims_to_abstract_ratio", "description_to_claims_ratio",
    "log_abstract_to_title_ratio", "log_claims_to_abstract_ratio", "log_description_to_claims_ratio",
    "sim_claims_abstract", "dist_claims_abstract",
    "claims_strategy_proxy",
]

df[new_text_features].describe(include="all")

,title_len,abstract_len,claims_len,description_len,log_title_len,log_abstract_len,log_claims_len,log_description_len,abstract_to_title_ratio,claims_to_abstract_ratio,description_to_claims_ratio,log_abstract_to_title_ratio,log_claims_to_abstract_ratio,log_description_to_claims_ratio,sim_claims_abstract,dist_claims_abstract,claims_strategy_proxy
count,200.000000,200.00000,200.00000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,63.875000,803.97000,5056.95500,19919.425000,4.025212,6.584852,8.413783,9.898498,16.762521,7.626982,5.172932,2.672132,2.001457,1.707308,0.774540,0.225460,0.460096
std,34.974641,366.59931,2286.22357,768.315369,0.561395,0.492702,0.515583,0.048463,12.995557,7.298459,3.792565,0.634011,0.508922,0.432216,0.159392,0.159392,0.354589
min,9.000000,57.00000,799.00000,12183.000000,2.302585,4.060443,6.684612,9.407879,0.387755,0.666945,1.666667,0.327687,0.510993,0.980829,0.320993,0.007520,0.010433
25%,38.000000,540.50000,3670.50000,20000.000000,3.663562,6.294280,8.208352,9.903538,8.888393,4.795266,3.253231,2.291358,1.757036,1.447678,0.670096,0.087107,0.170782
50%,54.500000,806.00000,4699.50000,20000.000000,4.016342,6.693304,8.455424,9.903538,12.590763,6.171853,4.179749,2.609100,1.970163,1.644687,0.785029,0.214971,0.393515
75%,84.250000,947.50000,6113.50000,20000.000000,4.445575,6.854881,8.718415,9.903538,19.840077,8.661321,5.436261,3.036877,2.268130,1.861948,0.912893,0.329904,0.680177
max,202.000000,2941.00000,12000.00000,20000.000000,5.313206,7.986845,9.392745,9.903538,96.200000,81.315789,25.030038,4.576771,4.410563,3.259251,0.992480,0.679007,1.527281


### Features al modelo
Para el modelo fional introduciremos `log_claims_to_abstract_ratio` y `sim_claims_abstract`

## Selección de variables NLP basadas en texto

Aunque se han generado múltiples variables derivadas del texto (longitudes, ratios y medidas de similitud), no todas se incorporan al modelo final. La selección se ha realizado siguiendo un criterio de **parsimonia**, priorizando variables que aportan información complementaria y evitan redundancias.

En particular:

- Las **longitudes absolutas de texto** (título, abstract, claims, descripción), así como sus transformaciones logarítmicas, se descartan por ser altamente correlacionadas entre sí y no aportar señal incremental relevante sobre el ROI.
- De los **ratios de longitud**, se conserva únicamente `log_claims_to_abstract_ratio`, ya que captura la complejidad jurídica relativa de las reivindicaciones frente al núcleo técnico descrito en el abstract.
- De las variables de **coherencia semántica entre bloques**, se conserva `sim_claims_abstract`, que mide el grado de alineación entre el contenido técnico del abstract y la estrategia de protección reflejada en las claims.

De este modo, el bloque NLP basado en estructura y redacción del texto queda reducido a un conjunto compacto, interpretable y estable bajo regularización, complementando las variables semánticas derivadas de embeddings.


# Executive Summary — NLP Feature Engineering Notebook

This notebook builds compact and interpretable NLP features from patent text to complement the existing structural ROI baseline. Instead of using raw text directly, it relies on multilingual sentence embeddings to derive semantic signals that are robust and easy to integrate into regularized tabular models.

Main steps:
- Generate multilingual embeddings (mainly from abstracts).
- Create low-dimensional semantic features:
  - `semantic_novelty`: distance to the global semantic centroid (proxy for semantic rarity).
  - `sim_to_high_roi`: cosine similarity to the centroid of high-ROI patents (proxy for semantic alignment with historically profitable areas).
- Add interaction features that capture contextual effects:
  - `novelty_x_family`: novelty scaled by international strategy (family size).
  - `novelty_x_breadth`: novelty scaled by technological breadth (CPC diversity proxy).
- Extend text-based features with claim/abstract structure:
  - `log_claims_to_abstract_ratio`: relative legal/technical verbosity.
  - `sim_claims_abstract`: semantic coherence between claims and abstract.

EDA results show that raw semantic features are typically weak-to-moderate on their own, while interaction features (novelty × family/breadth) often show clearer monotonic structure vs ROI, making them strong candidates for a final regularized model. The notebook concludes with a parsimonious feature selection strategy: keep only a small set of non-redundant NLP variables that add incremental signal over the structural baseline.


# Resumen Ejecutivo — Cuaderno de Enriquecimiento NLP

Este cuaderno construye variables NLP compactas e interpretables a partir del texto de patentes para complementar el baseline estructural de ROI. En lugar de usar texto bruto, se emplean embeddings multilingües para derivar señales semánticas robustas que se integran bien en modelos tabulares regularizados.

Pasos principales:
- Generación de embeddings multilingües (principalmente del abstract).
- Construcción de variables semánticas de baja dimensionalidad:
  - `semantic_novelty`: distancia al centroide semántico global (proxy de rareza/novedad).
  - `sim_to_high_roi`: similitud coseno al centroide de patentes de alto ROI (proxy de alineación con áreas rentables).
- Creación de interacciones que capturan efectos contextuales:
  - `novelty_x_family`: novedad escalada por estrategia internacional (tamaño de familia).
  - `novelty_x_breadth`: novedad escalada por amplitud tecnológica (diversidad CPC).
- Extensión con señales de estructura textual (claims/abstract):
  - `log_claims_to_abstract_ratio`: complejidad jurídica relativa frente al núcleo técnico.
  - `sim_claims_abstract`: coherencia semántica entre abstract y reivindicaciones.

Los resultados EDA indican que las variables semánticas simples suelen ser débiles o moderadas, mientras que las interacciones (novedad × familia/amplitud) tienden a mostrar relaciones más claras con el ROI. El cuaderno termina con una estrategia de selección parsimoniosa: conservar solo un conjunto pequeño de variables NLP no redundantes que aporten señal incremental al modelo estructural.
